# Layer 3 · Step 0: the pairs and the checks

Step 0 of `Essential_task_delegation/Layer3_Task/Layer3_Task_Brief.md`, built 18 Nov 2026. **CPU only:** set the accelerator to None, so it uses none of the weekly GPU hours.

**What it does, and why**
- Every later Layer 3 step reads one file, `pairs.csv`, so the training and validation pairs can't drift between steps.
- It rebuilds the 6,443 pairs and the D4 split exactly as every D4 run did:
  - R11's Cell 4, word for word except the mask folder, as in E5 steps 1 and 2;
  - R11's split line, `train_test_split(pairs, test_size=0.1, random_state=42)`.
- It checks by image content, not by file name, that no validation photo is also a training photo:
  - it hashes the decoded pixels with E5 step 1's code;
  - it checks all four combinations: a validation before-photo against training before-photos and training after-photos, and a validation after-photo against both. E5 step 1 checked two of them (before against before, after against after).
- **Why it matters:** on 26 Oct, validation scores turned out to be inflated by 40 pairs that share a before-photo with a training pair (socal_fire and woolsey_fire). For Layer 3 a leak matters even more. The generator learns to output training before-photos, so it could simply remember a validation pair's before-photo.

**The checks.** The last cell prints all seven. If any fails, it then stops with an error, and the printout goes to Abrar.
1. 6,443 pairs, each with a before-photo.
2. 5,798 training and 645 validation pairs, 560 of them from xBD.
3. 1,808,111 destroyed pixels in the validation masks, so the masks are the ones every D4 run scored.
4. E5's 40 leaked pairs are found again.
5. Nothing new leaks: no other validation pair shares a photo with a training pair, in any combination.
6. 605 clean validation pairs.
7. Every photo and every mask can be read.

**Expected, fixed before the run (18 Nov)**
- A check on the PC on 18 Nov used the pixel hashes that E5 step 1 saved (`Champion_Task/E5-results/step1/fingerprints_instruct.csv`). It found:
  - 5,798 training and 645 validation pairs (560 xBD);
  - exactly E5's 40 leaked pairs, all of them "validation before-photo = training before-photo", holding 155,812 destroyed pixels;
  - no match in the two combinations E5 didn't check;
  - no pair whose before- and after-photo are the same picture;
  - 605 clean pairs: 520 from xBD and 85 others, 64 of them from Turkey.
- So this run should pass all seven checks with these numbers.
- **Near copies** are listed, never counted. E5's rule: dHash within 6 of 64 bits, and 32 × 32 grey thumbnails correlating at 0.95 or more.
  - Among before-photos, E5 found 0 near copies and 3 matches for review.
  - For the other combinations, no number is predicted. The after-photos of the 40 leaked pairs should be among them: E5 found that they look alike (median correlation 0.97).

**How to run on Kaggle (your own account)**
1. Session options: Accelerator **None**. Internet isn't needed.
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. Smoke test, about 10 minutes: Run All with `SMOKE_TEST = True`.
   - It builds the full split, but fingerprints only a sample: the 645 validation pairs, the socal_fire and woolsey_fire training pairs (all 40 of E5's twins are there) and 200 random training pairs.
   - It must find all 40. Send its printout to Abrar.
4. After his go, the full run, about 20 minutes: set `SMOKE_TEST = False`, then Save Version → Save & Run All (Commit).

**Outputs**, in `/kaggle/working/layer3-step0/`
- `pairs.csv`: one row per pair (6,443), in the D4 order:
  - `after_image`, `before_image`: file names in `DisasterM3_Instruct/train_images/train_images/`;
  - `mask_name`, `classes`: the true mask's binary masks are `DisasterM3_Instruct/masks/masks/train_building_<intact|damaged|destroyed>_mask/<mask_name>`, for the classes listed (1 intact, 2 damaged, 3 destroyed). R11's Cell 4 builds the 4-class mask from them;
  - `split` (train or val), `clean` (validation rows only: False for E5's 40), `event` and `source` (E1's rule);
  - `height`, `width`, `before_same_size`, the 4-class mask's pixels per class (`background_px` to `destroyed_px`), and each photo's pixel hash.
- `leak_check.csv`: every validation photo that matches a training photo: the combination, `same pixels` (counted) or a near copy (listed only), and whether the pair is one of E5's 40.
- `summary.json`: the counts, the checks, the library versions and the run time.

In [1]:
# ── Cell 1: Settings ──────────────────────────────────────────────────────
import hashlib
import json
import os
import platform
import random
import re
import shutil
import time
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import sklearn

SMOKE_TEST = False                      # True = fingerprint a sample (about 10 min); False = every photo (about 20 min)
SEED = 42                              # for the smoke test's sample (the split has its own random_state=42)

# ── Paths: R11's Cell 2 (the same DisasterM3 mirror dataset) ──
DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/layer3-step0")
MASK_TMP = "/tmp/combined_masks"       # where R11's Cell 4 writes the 4-class masks (as in E5 steps 1 and 2)
WORKERS = os.cpu_count() or 4

NUM_CLASSES = 4                        # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]

# ── E5 step 1's thresholds: a near copy is listed, never counted ──
NEAR_DHASH, NEAR_CORR = 6, 0.95        # a near copy: dHash within 6 of 64 bits AND grey thumbnails correlating >= 0.95
REVIEW_DHASH, REVIEW_CORR = 10, 0.85   # looser: listed for review only
DARK, FLAT = 0.5, 4.0                  # never near-matched: images over half black, or thumbnails with grey std < 4

# ── The expectation (Cell 0) ──
EXPECTED = {"pairs": 6443, "train": 5798, "val": 645, "val_xbd": 560, "val_destroyed_px": 1808111,
            "leaked": 40, "leaked_destroyed_px": 155812, "clean": 605}
# E5 step 1's 40 leaked validation pairs, by after-photo: the rows of
# Champion_Task/E5-results/step1/val_train_leaks_hash.csv with `counted` True
E5_LEAKED_40 = [
    "socal_fire_00000081_post_disaster.png", "socal_fire_00000188_post_disaster.png",
    "socal_fire_00000218_post_disaster.png", "socal_fire_00000220_post_disaster.png",
    "socal_fire_00000222_post_disaster.png", "socal_fire_00000235_post_disaster.png",
    "socal_fire_00000301_post_disaster.png", "socal_fire_00000341_post_disaster.png",
    "socal_fire_00000348_post_disaster.png", "socal_fire_00000396_post_disaster.png",
    "socal_fire_00000436_post_disaster.png", "socal_fire_00000455_post_disaster.png",
    "socal_fire_00000497_post_disaster.png", "socal_fire_00000504_post_disaster.png",
    "socal_fire_00000614_post_disaster.png", "socal_fire_00001188_post_disaster.png",
    "woolsey_fire_00000009_post_disaster.png", "woolsey_fire_00000030_post_disaster.png",
    "woolsey_fire_00000032_post_disaster.png", "woolsey_fire_00000137_post_disaster.png",
    "woolsey_fire_00000272_post_disaster.png", "woolsey_fire_00000399_post_disaster.png",
    "woolsey_fire_00000435_post_disaster.png", "woolsey_fire_00000474_post_disaster.png",
    "woolsey_fire_00000498_post_disaster.png", "woolsey_fire_00000514_post_disaster.png",
    "woolsey_fire_00000529_post_disaster.png", "woolsey_fire_00000573_post_disaster.png",
    "woolsey_fire_00000576_post_disaster.png", "woolsey_fire_00000589_post_disaster.png",
    "woolsey_fire_00000623_post_disaster.png", "woolsey_fire_00000632_post_disaster.png",
    "woolsey_fire_00000634_post_disaster.png", "woolsey_fire_00000653_post_disaster.png",
    "woolsey_fire_00000679_post_disaster.png", "woolsey_fire_00000703_post_disaster.png",
    "woolsey_fire_00000722_post_disaster.png", "woolsey_fire_00000758_post_disaster.png",
    "woolsey_fire_00000783_post_disaster.png", "woolsey_fire_00000810_post_disaster.png",
]
assert len(set(E5_LEAKED_40)) == EXPECTED["leaked"], "E5's list must hold 40 different pairs"

if not MANIFEST_PATH.exists():         # the mount path can differ between accounts: look for the manifest
    _found = sorted(Path("/kaggle/input").glob("**/DisasterM3_Instruct/train_release.json"))
    assert _found, "No manifest: attach the dataset abrarmohammedtanzim/disasterm3-mirror"
    DATA_ROOT, MANIFEST_PATH = _found[0].parent, _found[0]
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "opencv": cv2.__version__,
            "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
T_START = time.time()
print(f"✓ Layer 3 step 0{'  [SMOKE TEST]' if SMOKE_TEST else ''}: data {DATA_ROOT}")
print(f"  Outputs {OUT_DIR}; {WORKERS} worker(s); versions {VERSIONS}")

✓ Layer 3 step 0: data /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct
  Outputs /kaggle/working/layer3-step0; 4 worker(s); versions {'python': '3.13.18', 'numpy': '2.1.3', 'opencv': '4.14.0', 'pandas': '2.3.3', 'scikit-learn': '1.6.1'}


## 1. The pairs and the split
- **Cell 2** is R11's Cell 4, word for word except the mask folder (`MASK_TMP`), as in E5 steps 1 and 2.
- **Cell 3** is R11's split line. Then it builds one row per pair for `pairs.csv`:
  - the event and source (E1's rule, copied from E5 step 1);
  - the true mask's name and its pixels per class;
  - whether a validation pair is clean.

In [2]:
# ── Cell 2: R11's Cell 4, word for word except the mask folder (MASK_TMP), as in E5 steps 1 and 2 ──
# ── Cell 4: Load manifest and extract segmentation entries ────────────────
# Mirrors train_qwen_disasterm3.ipynb Cell 4: same manifest, same path
# resolution logic, but this time we KEEP the segmentation entries and
# DROP the text-QA entries (exact inverse of the VLM notebook).
# ── Cell 4 (rewritten): Build combined multi-class masks from 3 binary sources ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs(MASK_TMP, exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"{MASK_TMP}/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Before: {pairs[0]['pre_image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Before: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_pre_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [3]:
# ── Cell 3: The split (R11's split line), and one row per pair for pairs.csv ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)   # R11's Cell 5, word for word
missing_pre = sum(p["pre_image_path"] is None for p in pairs)

# E1's event rule and a file-name helper: E5 step 1's Cell 3, word for word
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

masks_of = {name_of(k): v for k, v in by_image.items()}   # after-photo name -> its binary masks (Cell 2)
split_of = {**{name_of(p["image_path"]): "train" for p in train_pairs},
            **{name_of(p["image_path"]): "val" for p in val_pairs}}
assert len(split_of) == len(pairs), "two pairs share an after-photo name"
LEAKED_40 = set(E5_LEAKED_40)

ROWS = []                                                  # one row per pair, in the D4 order
for i, p in enumerate(pairs):
    after = name_of(p["image_path"])
    before = name_of(p["pre_image_path"]) if p["pre_image_path"] else ""
    m = masks_of[after]
    mask_names = sorted({name_of(m[f]) for f in FOLDER_TO_CLASS if f in m})
    gt = cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE)
    px = np.bincount(gt.ravel(), minlength=NUM_CLASSES) if gt is not None else np.zeros(NUM_CLASSES, np.int64)
    event, source = event_and_source(after)
    split = split_of[after]
    ROWS.append({"pair_index": i, "after_image": after, "before_image": before, "mask_name": ";".join(mask_names),
                 "classes": "+".join(str(c) for f, c in FOLDER_TO_CLASS.items() if f in m),
                 "split": split, "clean": (after not in LEAKED_40) if split == "val" else None,
                 "event": event, "source": source,
                 **{f"{n.lower()}_px": int(px[c]) for c, n in enumerate(CLASS_NAMES)},
                 "mask_ok": gt is not None and len(px) == NUM_CLASSES and len(mask_names) == 1,
                 "_after_path": p["image_path"], "_before_path": p["pre_image_path"]})

TRAIN = [r for r in ROWS if r["split"] == "train"]
VAL = [r for r in ROWS if r["split"] == "val"]
CLEAN = [r for r in VAL if r["clean"]]
n_xbd = sum(r["source"] == "xBD" for r in VAL)
val_destroyed = sum(r["destroyed_px"] for r in VAL)
masks_bad = [r["after_image"] for r in ROWS if not r["mask_ok"]]
print(f"✓ Pairs {len(ROWS):,}: training {len(TRAIN):,}, validation {len(VAL)} ({n_xbd} xBD, {len(VAL) - n_xbd} other); "
      f"without a before-photo: {missing_pre}")
print(f"  Destroyed pixels in the validation masks: {val_destroyed:,} (every D4 run: {EXPECTED['val_destroyed_px']:,})")
print(f"  E5's 40 leaked pairs in this validation split: {sum(r['after_image'] in LEAKED_40 for r in VAL)} of 40; "
      f"clean validation pairs: {len(CLEAN)} ({sum(r['source'] == 'xBD' for r in CLEAN)} xBD, "
      f"{sum(r['source'] == 'other' for r in CLEAN)} other; "
      f"{sum(r['event'] == 'turkey_earthquake' for r in CLEAN)} from Turkey)")
print(f"  Masks unreadable, with a value above 3, or under two names: {len(masks_bad)}")

✓ Pairs 6,443: training 5,798, validation 645 (560 xBD, 85 other); without a before-photo: 0
  Destroyed pixels in the validation masks: 1,808,111 (every D4 run: 1,808,111)
  E5's 40 leaked pairs in this validation split: 40 of 40; clean validation pairs: 605 (520 xBD, 85 other; 64 from Turkey)
  Masks unreadable, with a value above 3, or under two names: 0


## 2. A fingerprint of every photo
Cell 4 is E5 step 1's code, word for word: `fingerprint`, the self-checks on made-up images, `fingerprint_all`, `as_table` and `candidates`.
- **Pixel MD5:** a hash of the decoded pixels, so a copy saved again with other file bytes still matches. This is the counted rule.
- **dHash and a 32 × 32 grey thumbnail:** they find near copies (recompressed, slightly shifted or cropped). These are only listed.

Only the photos of the 6,443 pairs are fingerprinted. E5 also fingerprinted every other DisasterM3 file and the benchmark, which this step doesn't need.

In [4]:
# ── Cell 4: Fingerprints: E5 step 1's Cell 4, word for word, for the photos of our pairs only ──
import multiprocessing as mp

def fingerprint(path):
    """One image file -> its fingerprints. Never raises: an unreadable file gets an empty pixel hash."""
    cv2.setNumThreads(1)
    raw = Path(path).read_bytes()
    out = {"path": str(path), "file_md5": hashlib.md5(raw).hexdigest(), "pixel_md5": "", "dhash": 0,
           "dark": 1.0, "h": 0, "w": 0, "thumb": np.zeros((32, 32), np.uint8)}
    img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR) if raw else None
    if img is None:
        return out
    h, w = img.shape[:2]
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(grey, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    bits = (small[:, 1:] > small[:, :-1]).ravel()
    out.update(pixel_md5=hashlib.md5(f"{h}x{w}:".encode() + img.tobytes()).hexdigest(),
               dhash=int.from_bytes(np.packbits(bits).tobytes(), "big"),
               dark=round(float((grey < 8).mean()), 4), h=h, w=w,
               thumb=cv2.resize(grey, (32, 32), interpolation=cv2.INTER_AREA))
    return out

POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)

def bit_distance(a, b):
    """Hamming distance between uint64 values or arrays (broadcast)."""
    x = np.bitwise_xor(np.asarray(a, np.uint64), np.asarray(b, np.uint64))
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POPCOUNT8[np.ascontiguousarray(x).view(np.uint8).reshape(*x.shape, 8)].sum(-1).astype(np.int16)

def thumb_corr(a, b):
    """Pearson correlation of paired rows of a and b (flattened thumbnails)."""
    a = a - a.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a * a).sum(1) * (b * b).sum(1))
    return np.where(den > 0, (a * b).sum(1) / np.maximum(den, 1e-9), 0.0)

def tier(d, c):
    """A pair's class: a near-copy is counted; a review match is only pictured."""
    if d <= NEAR_DHASH and c >= NEAR_CORR:
        return "near copy"
    return "review" if c >= REVIEW_CORR else None

# Self-checks on made-up images: the same pixels saved again, a recompressed copy, a 3-pixel shift, an unrelated image
def _texture(seed, size=512):
    rng = np.random.default_rng(seed)
    t = cv2.GaussianBlur(rng.integers(0, 256, (size, size, 3)).astype(np.float32), (0, 0), 6)
    return cv2.normalize(t, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

_chk = OUT_DIR / "_selfcheck"
_chk.mkdir(exist_ok=True)
_a = _texture(1)
cv2.imwrite(str(_chk / "a.png"), _a, [cv2.IMWRITE_PNG_COMPRESSION, 1])
cv2.imwrite(str(_chk / "a_again.png"), _a, [cv2.IMWRITE_PNG_COMPRESSION, 9])     # same pixels, other bytes
cv2.imwrite(str(_chk / "a_jpeg.jpg"), _a, [cv2.IMWRITE_JPEG_QUALITY, 85])          # recompressed
cv2.imwrite(str(_chk / "a_shift.png"), np.roll(_a, 3, axis=1))                     # shifted by 3 px
cv2.imwrite(str(_chk / "b.png"), _texture(2))                                      # unrelated
_f = {p.stem: fingerprint(p) for p in sorted(_chk.iterdir())}
shutil.rmtree(_chk)
_d = lambda x, y: int(bit_distance(_f[x]["dhash"], _f[y]["dhash"]))
_c = lambda x, y: float(thumb_corr(_f[x]["thumb"].reshape(1, -1).astype(np.float32),
                                   _f[y]["thumb"].reshape(1, -1).astype(np.float32))[0])
assert _f["a"]["pixel_md5"] == _f["a_again"]["pixel_md5"] and _f["a"]["file_md5"] != _f["a_again"]["file_md5"]
assert tier(_d("a", "a_jpeg"), _c("a", "a_jpeg")) == "near copy", (_d("a", "a_jpeg"), _c("a", "a_jpeg"))
assert tier(_d("a", "a_shift"), _c("a", "a_shift")) == "near copy", (_d("a", "a_shift"), _c("a", "a_shift"))
assert tier(_d("a", "b"), _c("a", "b")) is None and _d("a", "b") > REVIEW_DHASH, (_d("a", "b"), _c("a", "b"))
print(f"✓ Fingerprint self-checks passed (dHash: recompressed {_d('a', 'a_jpeg')}, shifted {_d('a', 'a_shift')}, "
      f"unrelated {_d('a', 'b')} of 64 bits)")

def fingerprint_all(files, label):
    out, t0 = [], time.time()
    # Workers started by "fork" can call fingerprint(), which lives in this notebook; Python 3.14's default
    # start method on Linux ("forkserver") can't, so fork is asked for by name.
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
    ex = ProcessPoolExecutor(max_workers=WORKERS, mp_context=ctx) if WORKERS > 1 else None
    results = ex.map(fingerprint, files, chunksize=16) if ex else map(fingerprint, files)
    for i, fp in enumerate(results, 1):
        out.append(fp)
        if i % 2000 == 0 or i == len(files):
            rate = i / max(time.time() - t0, 1e-6)
            print(f"  {label}: {i:,}/{len(files):,} ({rate:.0f} files/s; {(len(files) - i) / rate / 60:.1f} min left)")
    if ex:
        ex.shutdown()
    return out

def as_table(fps):
    T = {"names": [Path(f["path"]).name for f in fps], "paths": [f["path"] for f in fps],
         "pixel": [f["pixel_md5"] for f in fps], "file": [f["file_md5"] for f in fps],
         "dhash": np.array([f["dhash"] for f in fps], np.uint64),
         "dark": np.array([f["dark"] for f in fps], np.float32),
         "thumb": (np.stack([f["thumb"].reshape(-1) for f in fps]).astype(np.float32) if fps
                   else np.zeros((0, 1024), np.float32)),
         "h": [f["h"] for f in fps], "w": [f["w"] for f in fps]}
    T["matchable"] = (T["dark"] <= DARK) & (T["thumb"].std(1) >= FLAT) & np.array([bool(p) for p in T["pixel"]], bool)
    return T

def candidates(A, B, rows=None, cols=None, max_bits=REVIEW_DHASH, chunk=256):
    """All (i, j, dHash distance, thumbnail correlation) with distance <= max_bits, both images matchable."""
    ia = np.arange(len(A["names"])) if rows is None else np.asarray(rows, int)
    jb = np.arange(len(B["names"])) if cols is None else np.asarray(cols, int)
    ia, jb = ia[A["matchable"][ia]], jb[B["matchable"][jb]]
    hi, hj, hd = [], [], []
    for s in range(0, len(ia), chunk):
        blk = ia[s:s + chunk]
        d = bit_distance(A["dhash"][blk][:, None], B["dhash"][jb][None, :])
        ii, jj = np.nonzero(d <= max_bits)
        hi.append(blk[ii])
        hj.append(jb[jj])
        hd.append(d[ii, jj])
    i, j, d = (np.concatenate(h).astype(int) if h else np.zeros(0, int) for h in (hi, hj, hd))
    c = thumb_corr(A["thumb"][i], B["thumb"][j]) if len(i) else np.zeros(0)
    return i, j, d, c

# ── The photos: every photo of every pair. The smoke test takes a sample that holds all of E5's 40 twins ──
if SMOKE_TEST:
    _fire = [r for r in TRAIN if r["event"] in ("socal_fire", "woolsey_fire")]
    _rest = [r for r in TRAIN if r["event"] not in ("socal_fire", "woolsey_fire")]
    SAMPLE = VAL + _fire + random.Random(SEED).sample(_rest, 200)
else:
    SAMPLE = ROWS
PHOTOS = sorted({r[k] for r in SAMPLE for k in ("_after_path", "_before_path") if r[k]})
t0 = time.time()
FP = as_table(fingerprint_all(PHOTOS, "Photos"))
FP_MIN = (time.time() - t0) / 60
k_of = {path: k for k, path in enumerate(FP["paths"])}
UNREADABLE = sorted(FP["names"][k] for k, h in enumerate(FP["pixel"]) if not h)
print(f"✓ Fingerprinted {len(PHOTOS):,} photos of {len(SAMPLE):,} pairs in {FP_MIN:.1f} min; "
      f"unreadable: {len(UNREADABLE)}; never near-matched (dark, flat or unreadable): {int((~FP['matchable']).sum()):,}")

✓ Fingerprint self-checks passed (dHash: recompressed 0, shifted 1, unrelated 32 of 64 bits)
  Photos: 2,000/12,886 (28 files/s; 6.6 min left)
  Photos: 4,000/12,886 (28 files/s; 5.3 min left)
  Photos: 6,000/12,886 (28 files/s; 4.1 min left)
  Photos: 8,000/12,886 (28 files/s; 3.0 min left)
  Photos: 10,000/12,886 (28 files/s; 1.7 min left)
  Photos: 12,000/12,886 (27 files/s; 0.5 min left)
  Photos: 12,886/12,886 (27 files/s; 0.0 min left)
✓ Fingerprinted 12,886 photos of 6,443 pairs in 7.8 min; unreadable: 0; never near-matched (dark, flat or unreadable): 308


## 3. Does any validation photo appear in training?
- **Counted:** a validation photo (before or after) with the same pixels as any training photo (before or after).
- **Listed only:** each validation photo's closest near copy among the training photos, by E5's thresholds.
- **For information:** pairs whose before-photo and after-photo are the same picture.

In [5]:
# ── Cell 5: Validation photos that are also training photos, in all four combinations ──
ROLES = (("before", "_before_path"), ("after", "_after_path"))
uses = defaultdict(list)                       # photo index -> [(pair row, its role)]
for r in SAMPLE:
    for role, key in ROLES:
        if k_of.get(r[key]) is not None:
            uses[k_of[r[key]]].append((r, role))
train_by_pixel = defaultdict(list)             # pixel MD5 -> [(training pair, role)]
for k, us in uses.items():
    for r, role in us:
        if FP["pixel"][k] and r["split"] == "train":
            train_by_pixel[FP["pixel"][k]].append((r, role))

def leak_row(r, role, t, t_role, match, d, c):
    return {"val_after_image": r["after_image"], "val_before_image": r["before_image"], "event": r["event"],
            "val_photo": role, "train_photo": t_role, "train_pair": t["after_image"],
            "train_file": t["before_image"] if t_role == "before" else t["after_image"],
            "match": match, "dhash": d, "corr": c, "counted": match == "same pixels",
            "in_e5_40": r["after_image"] in LEAKED_40}

# Counted: the same pixels
LEAK_ROWS = []
for k, us in uses.items():
    h = FP["pixel"][k]
    for r, role in us:
        if h and r["split"] == "val":
            LEAK_ROWS.extend(leak_row(r, role, t, t_role, "same pixels", 0, 1.0) for t, t_role in train_by_pixel.get(h, []))

# Listed only: each validation photo's closest near copy among the training photos (E5's thresholds)
val_k = sorted(k for k, us in uses.items() if any(r["split"] == "val" for r, _ in us))
train_k = sorted(k for k, us in uses.items() if any(r["split"] == "train" for r, _ in us))
best = {}
for i, j, d, c in zip(*candidates(FP, FP, rows=val_k, cols=train_k)):
    t = tier(d, c)
    if t and i != j and FP["pixel"][i] != FP["pixel"][j]:     # the same pixels are counted above
        key = (0 if t == "near copy" else 1, int(d), -float(c))
        if i not in best or key < best[i][0]:
            best[i] = (key, t, j, int(d), round(float(c), 4))
for i, (_, t, j, d, c) in best.items():
    for r, role in uses[i]:
        for tr, t_role in uses[j]:
            if r["split"] == "val" and tr["split"] == "train":
                LEAK_ROWS.append(leak_row(r, role, tr, t_role, t, d, c))

COUNTED = [x for x in LEAK_ROWS if x["counted"]]
LEAKED = {x["val_after_image"] for x in COUNTED}
FOUND_40 = LEAKED & LEAKED_40
MISSING_40 = sorted(LEAKED_40 - LEAKED)
NEW_LEAKS = sorted(LEAKED - LEAKED_40)
LEAKED_PX = sum(r["destroyed_px"] for r in VAL if r["after_image"] in LEAKED)
COMBOS = Counter(f"validation {x['val_photo']} = training {x['train_photo']}" for x in COUNTED)
NEAR = [x for x in LEAK_ROWS if not x["counted"]]
NEAR_CLEAN = sorted({x["val_after_image"] for x in NEAR if x["match"] == "near copy" and not x["in_e5_40"]})

def pixel_of(path):
    return FP["pixel"][k_of[path]] if k_of.get(path) is not None else ""

SAME_WITHIN = sorted(r["after_image"] for r in SAMPLE
                     if pixel_of(r["_after_path"]) and pixel_of(r["_after_path"]) == pixel_of(r["_before_path"]))

print(f"✓ Validation pairs sharing a photo with a training pair (same pixels, counted): {len(LEAKED)}, "
      f"holding {LEAKED_PX:,} destroyed pixels")
for combo in ("validation before = training before", "validation after = training after",
              "validation before = training after", "validation after = training before"):
    print(f"  {combo}: {COMBOS.get(combo, 0)} match(es)")
print(f"  E5's 40 found again: {len(FOUND_40)} of 40"
      f"{(' (missing: ' + ', '.join(MISSING_40) + ')') if MISSING_40 else ''}; new leaked pairs: {len(NEW_LEAKS)}"
      f"{(' (' + ', '.join(NEW_LEAKS[:10]) + ')') if NEW_LEAKS else ''}")
print(f"  Near copies (listed, not counted): {sum(x['match'] == 'near copy' for x in NEAR)} "
      f"({sum(x['match'] == 'near copy' and x['in_e5_40'] for x in NEAR)} in E5's 40 pairs); "
      f"review: {sum(x['match'] == 'review' for x in NEAR)}; clean validation pairs with a near copy: {len(NEAR_CLEAN)}")
print(f"  Pairs whose before- and after-photo are the same picture: {len(SAME_WITHIN)}")

✓ Validation pairs sharing a photo with a training pair (same pixels, counted): 40, holding 155,812 destroyed pixels
  validation before = training before: 40 match(es)
  validation after = training after: 0 match(es)
  validation before = training after: 0 match(es)
  validation after = training before: 0 match(es)
  E5's 40 found again: 40 of 40; new leaked pairs: 0
  Near copies (listed, not counted): 23 (23 in E5's 40 pairs); review: 28; clean validation pairs with a near copy: 0
  Pairs whose before- and after-photo are the same picture: 0


In [6]:
# ── Cell 6: The output files, and the seven checks against the expectation ──
for r in ROWS:
    ka, kb = k_of.get(r["_after_path"]), k_of.get(r["_before_path"])
    seen = ka is not None and kb is not None
    r["height"], r["width"] = (FP["h"][ka], FP["w"][ka]) if seen else (None, None)
    r["before_same_size"] = ((FP["h"][kb], FP["w"][kb]) == (FP["h"][ka], FP["w"][ka])) if seen else None
    r["after_pixel_md5"], r["before_pixel_md5"] = (FP["pixel"][ka], FP["pixel"][kb]) if seen else ("", "")
PAIR_COLS = ["pair_index", "after_image", "before_image", "mask_name", "classes", "split", "clean", "event", "source",
             "height", "width", "before_same_size", "background_px", "intact_px", "damaged_px", "destroyed_px",
             "after_pixel_md5", "before_pixel_md5"]
LEAK_COLS = ["val_after_image", "val_before_image", "event", "val_photo", "train_photo", "train_pair", "train_file",
             "match", "dhash", "corr", "counted", "in_e5_40"]
pd.DataFrame(ROWS)[PAIR_COLS].to_csv(OUT_DIR / "pairs.csv", index=False)
pd.DataFrame(sorted(LEAK_ROWS, key=lambda x: (not x["counted"], x["val_after_image"], x["val_photo"])),
             columns=LEAK_COLS).to_csv(OUT_DIR / "leak_check.csv", index=False)

in_sample = " (smoke test: within the sample only)" if SMOKE_TEST else ""
CHECKS = [
    ("1. 6,443 pairs, each with a before-photo",
     len(pairs) == EXPECTED["pairs"] and skipped_img == 0 and missing_pre == 0,
     f"{len(pairs):,} pairs; after-photos not found: {skipped_img}; without a before-photo: {missing_pre}"),
    ("2. 5,798 training and 645 validation pairs (560 xBD)",
     (len(TRAIN), len(VAL), n_xbd) == (EXPECTED["train"], EXPECTED["val"], EXPECTED["val_xbd"]),
     f"{len(TRAIN):,} training, {len(VAL)} validation ({n_xbd} xBD)"),
    ("3. 1,808,111 destroyed pixels in the validation masks",
     val_destroyed == EXPECTED["val_destroyed_px"], f"{val_destroyed:,}"),
    ("4. E5's 40 leaked pairs found again", not MISSING_40,
     f"{len(FOUND_40)} of 40; the leaked pairs hold {LEAKED_PX:,} destroyed pixels "
     f"(E5: {EXPECTED['leaked_destroyed_px']:,})"),
    ("5. Nothing new leaks" + in_sample, not NEW_LEAKS, f"new leaked pairs: {len(NEW_LEAKS)}"),
    ("6. 605 clean validation pairs", len(CLEAN) == EXPECTED["clean"], f"{len(CLEAN)} clean"),
    ("7. Every photo and every mask can be read" + in_sample, not UNREADABLE and not masks_bad,
     f"unreadable photos: {len(UNREADABLE)} of {len(PHOTOS):,}; bad masks: {len(masks_bad)} of {len(ROWS):,}"),
]

ev_all, ev_clean = Counter(r["event"] for r in VAL), Counter(r["event"] for r in CLEAN)
summary = {
    "task": "Layer 3 step 0", "smoke_test": SMOKE_TEST, "minutes": round((time.time() - T_START) / 60, 1),
    "fingerprint_minutes": round(FP_MIN, 1), "versions": VERSIONS,
    "pairs": len(pairs), "train": len(TRAIN), "val": len(VAL), "val_xbd": n_xbd,
    "val_destroyed_px": val_destroyed, "clean": len(CLEAN),
    "clean_by_source": dict(Counter(r["source"] for r in CLEAN)),
    "clean_turkey": sum(r["event"] == "turkey_earthquake" for r in CLEAN),
    "val_by_event": {e: {"pairs": ev_all[e], "clean": ev_clean.get(e, 0)} for e in sorted(ev_all)},
    "photos_fingerprinted": len(PHOTOS), "pairs_fingerprinted": len(SAMPLE),
    "before_photo_other_size": sum(r["before_same_size"] is False for r in ROWS),
    "unreadable_photos": UNREADABLE, "bad_masks": masks_bad,
    "leaks": {"counted_pairs": len(LEAKED), "destroyed_px": LEAKED_PX, "by_combination": dict(COMBOS),
              "e5_40_found": len(FOUND_40), "e5_40_missing": MISSING_40, "new": NEW_LEAKS},
    "near_copies": {"near_copy": sum(x["match"] == "near copy" for x in NEAR),
                    "near_copy_in_e5_40": sum(x["match"] == "near copy" and x["in_e5_40"] for x in NEAR),
                    "review": sum(x["match"] == "review" for x in NEAR),
                    "clean_pairs_with_a_near_copy": NEAR_CLEAN},
    "same_picture_within_a_pair": SAME_WITHIN,
    "thresholds": {"near_dhash": NEAR_DHASH, "near_corr": NEAR_CORR, "review_dhash": REVIEW_DHASH,
                   "review_corr": REVIEW_CORR, "dark": DARK, "flat_std": FLAT},
    "checks": {name: ("passed" if ok else "FAILED") for name, ok, _ in CHECKS},
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")

print(f"Layer 3 step 0{' [SMOKE TEST]' if SMOKE_TEST else ''}: the checks")
for name, ok, found in CHECKS:
    print(f"  {'✓' if ok else '✗'} {name}: {'passed' if ok else 'FAILED'} ({found})")
print(f"\n✓ Wrote {OUT_DIR}: pairs.csv, leak_check.csv, summary.json "
      f"({summary['minutes']} min in all, {FP_MIN:.1f} of them fingerprinting)")
failed = [name for name, ok, _ in CHECKS if not ok]
if failed:
    raise RuntimeError("STOP: send this printout to Abrar. Failed: " + "; ".join(failed))
print("✓ All seven checks passed" + (" on the smoke test's sample; the full run decides" if SMOKE_TEST else ""))

Layer 3 step 0: the checks
  ✓ 1. 6,443 pairs, each with a before-photo: passed (6,443 pairs; after-photos not found: 0; without a before-photo: 0)
  ✓ 2. 5,798 training and 645 validation pairs (560 xBD): passed (5,798 training, 645 validation (560 xBD))
  ✓ 3. 1,808,111 destroyed pixels in the validation masks: passed (1,808,111)
  ✓ 4. E5's 40 leaked pairs found again: passed (40 of 40; the leaked pairs hold 155,812 destroyed pixels (E5: 155,812))
  ✓ 5. Nothing new leaks: passed (new leaked pairs: 0)
  ✓ 6. 605 clean validation pairs: passed (605 clean)
  ✓ 7. Every photo and every mask can be read: passed (unreadable photos: 0 of 12,886; bad masks: 0 of 6,443)

✓ Wrote /kaggle/working/layer3-step0: pairs.csv, leak_check.csv, summary.json (31.3 min in all, 7.8 of them fingerprinting)
✓ All seven checks passed
